# Titanic - MAchine Learning from Disaster

The notebook gives solution for Titanic dataset question which trys to predict the survival of passengers based on different features.

### Import libraries

In [1]:
import numpy as np
import pandas as pd
from typing import List, Optional, Union
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns


## Read data

In [2]:
train_path = Path("data/train.csv")
test_path = Path("data/test.csv")

In [3]:
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

In [4]:
df_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
print(df_train.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')


In [6]:
df_test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [7]:
print(df_train.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')


## Data Analysis

In [8]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [9]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


In [10]:

print("Total missing values in train dataset:","\n",  df_train.isnull().sum() , "\n" ,'out of total rows:', len(df_train))

Total missing values in train dataset: 
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64 
 out of total rows: 891


In [11]:
print("Total missing values in test dataset:","\n",  df_test.isnull().sum() , "\n" ,'out of total rows:', len(df_test))

Total missing values in test dataset: 
 PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64 
 out of total rows: 418


In [12]:
import plotly.graph_objects as go
import pandas as pd

df = df_train.copy()

df["Sex_encoded"] = df["Sex"].map({"male": 0, "female": 1})

columns_to_try = {
    "Sex": "Sex_encoded",
    "Pclass": "Pclass",
    "Age": "Age",
    "Fare": "Fare",
    "SibSp": "SibSp",
    "Parch": "Parch",
    "Embarked": "Embarked",
    "Ticket" : "Ticket"

}

fig = go.Figure()

for i, (label, col) in enumerate(columns_to_try.items()):
    fig.add_trace(
        go.Scatter(
            x=df[col],
            y=df["PassengerId"],
            mode="markers",
            marker=dict(
                color=df["Survived"],
                colorscale="Viridis",
                showscale=True if i == 0 else False,
                colorbar=dict(title="Survived"),
            ),
            name=label,
            visible=True if i == 0 else False
        )
    )

buttons = []

for i, label in enumerate(columns_to_try.keys()):
    visible_array = [False] * len(columns_to_try)
    visible_array[i] = True

    buttons.append(
        dict(
            label=label,
            method="update",
            args=[
                {"visible": visible_array},
                {"xaxis": {"title": label}}
            ],
        )
    )

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
        )
    ],
    title="Survival Scatter Plot (Selectable X-Axis)",
    xaxis_title=list(columns_to_try.keys())[0],
    yaxis_title="PassengerId"
)

fig.show()

In [13]:
import pandas as pd
import plotly.graph_objects as go


def make_configurable_3d_scatter(
    df,
    axis_columns,
    color_column="Survived",
    default_x=None,
    default_y=None,
    default_z=None,
    colorscale="Viridis",
    encode_categoricals=True,
    title="3D Feature Explorer"
):
    df_plot = df.copy()

    # Normalize input
    if isinstance(axis_columns, list):
        axis_columns = {col: col for col in axis_columns}

    processed_cols = {}

    # Encode categoricals if needed
    for label, col in axis_columns.items():
        if not pd.api.types.is_numeric_dtype(df_plot[col]) and encode_categoricals:
            encoded_col = f"{col}__encoded"
            df_plot[encoded_col] = df_plot[col].astype("category").cat.codes
            processed_cols[label] = encoded_col
        else:
            processed_cols[label] = col

    labels = list(processed_cols.keys())

    # Defaults
    default_x = default_x or labels[0]
    default_y = default_y or labels[1]
    default_z = default_z or labels[2]

    x_col = processed_cols[default_x]
    y_col = processed_cols[default_y]
    z_col = processed_cols[default_z]

    # Initial figure
    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=df_plot[x_col],
            y=df_plot[y_col],
            z=df_plot[z_col],
            mode="markers",
            marker=dict(
                size=4,
                color=df_plot[color_column],
                colorscale=colorscale,
                opacity=0.8,
                colorbar=dict(title=color_column),
            ),
            name="data"
        )
    )

    # Dropdowns
    def make_axis_buttons(axis_name):
        buttons = []
        for label, col in processed_cols.items():
            update_args = {
                axis_name: [df_plot[col]]
            }

            layout_update = {
                f"scene.{axis_name}axis.title": label
            }

            buttons.append(
                dict(
                    label=label,
                    method="update",
                    args=[update_args, layout_update],
                )
            )
        return buttons

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title=default_x,
            yaxis_title=default_y,
            zaxis_title=default_z,
        ),
        template="plotly_white",
        height=700,
        updatemenus=[
            dict(
                buttons=make_axis_buttons("x"),
                direction="down",
                x=1.15,
                y=1.0,
                showactive=True,
            ),
            dict(
                buttons=make_axis_buttons("y"),
                direction="down",
                x=1.15,
                y=0.85,
                showactive=True,
            ),
            dict(
                buttons=make_axis_buttons("z"),
                direction="down",
                x=1.15,
                y=0.70,
                showactive=True,
            ),
        ],
        annotations=[
            dict(text="X Axis", x=1.12, y=1.05, showarrow=False),
            dict(text="Y Axis", x=1.12, y=0.90, showarrow=False),
            dict(text="Z Axis", x=1.12, y=0.75, showarrow=False),
        ]
    )

    return fig, df_plot

axis_columns = {
    "Pclass": "Pclass",
    "Age": "Age",
    "Fare": "Fare",
    "SibSp": "SibSp",
    "Parch": "Parch",
    "Sex": "Sex",
    "Embarked": "Embarked"
}

fig, df_processed = make_configurable_3d_scatter(
    df_train,
    axis_columns=axis_columns,
    color_column="Survived",
    default_x="Age",
    default_y="Fare",
    default_z="Pclass",
    title="Titanic 3D Feature Explorer"
)

fig.show()

In [14]:
df_train.Ticket.value_counts()

Ticket
347082             7
1601               7
CA. 2343           7
3101295            6
CA 2144            6
                  ..
SOTON/OQ 392076    1
211536             1
112053             1
111369             1
370376             1
Name: count, Length: 681, dtype: int64

## Data preprocessing 

In [16]:
df_train = pd.get_dummies(df_train, columns=["Sex", "Embarked" , "PassengerId" , "Name" , "Ticket"], drop_first=True)
df_test = pd.get_dummies(df_test, columns=["Sex", "Embarked" , "PassengerId" , "Name" , "Ticket"], drop_first=True)

In [17]:
df_train = df_train.drop(columns=["Cabin"])
df_test = df_test.drop(columns=["Cabin"])

In [18]:
df_train = df_train.dropna()
df_test = df_test.dropna()

In [19]:
df_train.isnull().sum()

Survived              0
Pclass                0
Age                   0
SibSp                 0
Parch                 0
                     ..
Ticket_W./C. 6608     0
Ticket_W./C. 6609     0
Ticket_W.E.P. 5734    0
Ticket_W/C 14208      0
Ticket_WE/P 5735      0
Length: 2469, dtype: int64

In [20]:
df_test.isnull().sum()

Pclass                0
Age                   0
SibSp                 0
Parch                 0
Fare                  0
                     ..
Ticket_W./C. 14260    0
Ticket_W./C. 14266    0
Ticket_W./C. 6607     0
Ticket_W./C. 6608     0
Ticket_W.E.P. 5734    0
Length: 1204, dtype: int64

In [23]:
import plotly.graph_objects as go
import pandas as pd

df = df_train.copy()



columns_to_try = {
    "Sex_male": "Sex_male",
    "Pclass": "Pclass",
    "Age": "Age",
    "Fare": "Fare",
    "SibSp": "SibSp",
    "Parch": "Parch",
    "Embarked_Q": "Embarked_Q",
    "Embarked_S": "Embarked_S",

}

fig = go.Figure()

for i, (label, col) in enumerate(columns_to_try.items()):
    fig.add_trace(
        go.Scatter(
            x=df[col],
            y=df["Sex_male"],
            mode="markers",
            marker=dict(
                color=df["Survived"],
                colorscale="Viridis",
                showscale=True if i == 0 else False,
                colorbar=dict(title="Survived"),
            ),
            name=label,
            visible=True if i == 0 else False
        )
    )

buttons = []

for i, label in enumerate(columns_to_try.keys()):
    visible_array = [False] * len(columns_to_try)
    visible_array[i] = True

    buttons.append(
        dict(
            label=label,
            method="update",
            args=[
                {"visible": visible_array},
                {"xaxis": {"title": label}}
            ],
        )
    )

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
        )
    ],
    title="Survival Scatter Plot (Selectable X-Axis)",
    xaxis_title=list(columns_to_try.keys())[0],
    yaxis_title="PassengerId"
)

fig.show()

## Random Forest Classification Model

## Logistic Regression Model

## Model Evaluation